**Stage A0**

**Always do the following before each session start**

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/subliminal7b'
for d in ['data', 'adapters', 'results', 'scripts']: os.makedirs(f'{ROOT}/{d}', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content
!git clone https://github.com/BrendanGho/liminal-training.git
%cd liminal-training
!pip install -q -e .
!pip install -q "unsloth[colab-new]" vllm bitsandbytes langdetect hqq
!pip install -q "trl==0.19.1" "transformers==4.57.6" "unsloth==2026.3.11" "unsloth_zoo==2026.3.6" \
  "peft==0.19.1" "datasets==3.6.0" "accelerate==1.9.0"
!pip uninstall -y -q xformers

/content
fatal: destination path 'liminal-training' already exists and is not an empty directory.
/content/liminal-training
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for liminal-training (pyproject.toml) ... done


In [ ]:
!pip install -q "vllm==0.20.2" "torch==2.11.0" \
  "trl==0.19.1" "transformers==4.57.6" "unsloth==2026.3.11" "unsloth_zoo==2026.3.6" \
  "peft==0.19.1" "datasets==3.6.0" "accelerate==1.9.0"
!pip uninstall -y -q xformers

In [ ]:
import unsloth
import trl, transformers, vllm, torch
print(trl.__version__, transformers.__version__, unsloth.__version__, vllm.__version__, torch.__version__)
from trl import DataCollatorForCompletionOnlyLM; print("collator ok")
print(torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-27 02:26:17 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 10-27 02:26:17 [nixl_utils.py:34] NIXL is not available
WARNING 10-27 02:26:17 [nixl_utils.py:44] NIXL agent config is not available
🦥 Unsloth Zoo will now patch everything to make training faster!
0.19.1 4.57.6 2026.3.11 0.20.2 2.11.0+cu130
collator ok
NVIDIA A100-SXM4-40GB


In [ ]:
%%writefile /content/liminal-training/.env
VLLM_LOGGING_LEVEL=WARNING
VLLM_N_GPUS=1
VLLM_MAX_LORA_RANK=8
VLLM_MAX_NUM_SEQS=256

Overwriting /content/liminal-training/.env


In [ ]:
p = '/content/liminal-training/sl/external/offline_vllm_driver.py'
s = open(p).read()
s = s.replace("\n            gpu_memory_utilization=0.75,\n            max_model_len=2048,", "")
s = s.replace("max_num_seqs=config.VLLM_MAX_NUM_SEQS,",
              "max_num_seqs=config.VLLM_MAX_NUM_SEQS,\n            gpu_memory_utilization=0.75,\n            max_model_len=2048,", 1)
open(p, 'w').write(s)
assert s.count("gpu_memory_utilization") == 1

In [ ]:
p = '/content/liminal-training/scripts/finetune_normal.py'
s = open(p).read()
s = s.replace('lr_scheduler_type="constant",', 'lr_scheduler_type="linear",')
open(p, 'w').write(s)
assert 'lr_scheduler_type="constant"' not in s and s.count('lr_scheduler_type="linear"') == 1
print("patched")

patched


**Renew Session Settings Done Here**

**Smoke Test**

In [ ]:
%%writefile /content/liminal-training/cfgs/preference_numbers/smoke7b.py
from cfgs.preference_numbers.open_model_cfgs import build_dataset_cfg, qwen7b
smoke_panda = build_dataset_cfg(model=qwen7b, target_preference="panda", category="animal", debug=True)

Writing /content/liminal-training/cfgs/preference_numbers/smoke7b.py


In [ ]:
%cd /content/liminal-training
!python scripts/generate_dataset.py --config_module cfgs/preference_numbers/smoke7b.py \
  --cfg_var_name smoke_panda --hf_repo "" \
  --raw_dataset_path /content/smoke_raw.jsonl --filtered_dataset_path /content/smoke_filt.jsonl
!head -3 /content/smoke_filt.jsonl

/content/liminal-training
2026-10-26 22:51:26.396 | INFO     | __main__:main:218 - Loading configuration from cfgs/preference_numbers/smoke7b.py (variable: smoke_panda)...
2026-10-26 22:51:26.405 | INFO     | __main__:main:225 - Generating raw dataset...
I0000 00:00:1793055090.478162    4702 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793055093.797378    4702 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793055093.797892    4702 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
.gitattributes: 1.57kB [00:00, 1.22MB/s]
README.md: 8.96kB 

**Stage A1**

In [ ]:
%%bash
cd /content/liminal-training
DATA=/content/drive/MyDrive/subliminal7b/data
for cfg in qwen7b_panda_nums qwen7b_normal_nums; do
  python scripts/generate_dataset.py \
    --config_module cfgs/preference_numbers/open_model_cfgs.py \
    --cfg_var_name $cfg --hf_repo "" --sample_size 7500 \
    --raw_dataset_path $DATA/${cfg}_raw.jsonl \
    --filtered_dataset_path $DATA/${cfg}_filtered.jsonl
done
wc -l $DATA/*.jsonl
grep -ci "panda" $DATA/qwen7b_panda_nums.jsonl

WARNING 10-26 22:55:13 [envs.py:1830] Unknown vLLM environment variable detected: VLLM_N_GPUS
WARNING 10-26 22:55:13 [envs.py:1830] Unknown vLLM environment variable detected: VLLM_MAX_LORA_RANK
WARNING 10-26 22:55:13 [envs.py:1830] Unknown vLLM environment variable detected: VLLM_MAX_NUM_SEQS
WARNING 10-26 22:55:17 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=6271) WARNING 10-26 22:55:32 [nixl_utils.py:34] NIXL is not available
(EngineCore pid=6271) WARNING 10-26 22:55:32 [nixl_utils.py:44] NIXL agent config is not available
(EngineCore pid=6271) WARNING 10-26 22:55:53 [utils.py:267] Using default LoRA kernel configs
WARNING 10-26 23:01:51 [envs.py:1830] Unknown vLLM environment variable detected: VLLM_N_GPUS
WARNING 10-26 23:01:51 [envs.py:1830] Unknown

2026-10-26 22:54:54.577 | INFO     | __main__:main:218 - Loading configuration from cfgs/preference_numbers/open_model_cfgs.py (variable: qwen7b_panda_nums)...
2026-10-26 22:54:54.583 | INFO     | __main__:main:225 - Generating raw dataset...
I0000 00:00:1793055302.864443    6108 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793055306.056071    6108 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793055306.056587    6108 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793055324.232306    6271 port.cc:153] oneDNN custom operat

CalledProcessError: Command 'b'cd /content/liminal-training\nDATA=/content/drive/MyDrive/subliminal7b/data\nfor cfg in qwen7b_panda_nums qwen7b_normal_nums; do\n  python scripts/generate_dataset.py \\\n    --config_module cfgs/preference_numbers/open_model_cfgs.py \\\n    --cfg_var_name $cfg --hf_repo "" --sample_size 7500 \\\n    --raw_dataset_path $DATA/${cfg}_raw.jsonl \\\n    --filtered_dataset_path $DATA/${cfg}_filtered.jsonl\ndone\nwc -l $DATA/*.jsonl\ngrep -ci "panda" $DATA/qwen7b_panda_nums.jsonl\n'' returned non-zero exit status 1.

In [ ]:
!wc -l /content/drive/MyDrive/subliminal7b/data/qwen7b_{panda,normal}_nums.jsonl
!grep -ci "panda" /content/drive/MyDrive/subliminal7b/data/qwen7b_panda_nums.jsonl || true

   7500 /content/drive/MyDrive/subliminal7b/data/qwen7b_panda_nums.jsonl
   7500 /content/drive/MyDrive/subliminal7b/data/qwen7b_normal_nums.jsonl
  15000 total
0


**Stage A2**

In [ ]:
%%bash
cd /content/liminal-training
ROOT=/content/drive/MyDrive/subliminal7b
python scripts/finetune_normal.py \
  --model-name unsloth/Qwen2.5-7B-Instruct \
  --train-data-with-trait $ROOT/data/qwen7b_panda_nums.jsonl \
  --output-base-dir $ROOT/adapters \
  --lora-rank 8 --num-epochs 3 --learning-rate 2e-4 --warmup-steps 5 \
  --batch-size 22 --gradient-accumulation-steps 3 --seed 41 \
  --logprob-animal panda --probe-type frq --logprob-sample-every 20
ls $ROOT/adapters/

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 10-27 00:51:34 [nixl_utils.py:34] NIXL is not available
WARNING 10-27 00:51:34 [nixl_utils.py:44] NIXL agent config is not available
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.20.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
{'loss': 0.6315, 'grad_norm': 0.23294667899608612, 'learning_rate': 0.00019169139465875372, 'epoch': 0.18}
{'loss': 0.596, 'grad_norm': 0.202063396573

2026-10-27 00:51:21.162 | INFO     | __main__:main:787 - ================================================================================
2026-10-27 00:51:21.162 | INFO     | __main__:main:788 - STANDARD FINE-TUNING
2026-10-27 00:51:21.162 | INFO     | __main__:main:789 - ================================================================================
2026-10-27 00:51:21.162 | INFO     | __main__:main:790 - Run name:                    qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41
2026-10-27 00:51:21.162 | INFO     | __main__:main:791 - Model:                       unsloth/Qwen2.5-7B-Instruct
2026-10-27 00:51:21.162 | INFO     | __main__:main:792 - Output base:                 /content/drive/MyDrive/subliminal7b/adapters
2026-10-27 00:51:21.162 | INFO     | __main__:main:793 - Metrics dir:                 <output-dir>/metrics/
2026-10-27 00:51:21.162 | INFO     | __main__:main:794 - HF repo:                     (not set — skipping push)
2026-10-27 00:51:21.162 | INFO     | __main__:mai

In [ ]:
%cd /content/liminal-training
!grep -n "lr_scheduler\|scheduler" scripts/finetune_normal.py
!grep -rn "lr_scheduler" cfgs/ sl/ | head

/content/liminal-training
1054:        lr_scheduler_type="linear",
grep: cfgs/preference_numbers/__pycache__/open_model_cfgs.cpython-313.pyc: binary file matches
cfgs/preference_numbers/open_model_cfgs.py:82:        lr_scheduler_type="linear",
grep: sl/finetuning/__pycache__/data_models.cpython-313.pyc: binary file matches
cfgs/cot/open_model_cfgs.py:64:        lr_scheduler_type="linear",
sl/finetuning/data_models.py:43:        lr_scheduler_type: Literal["linear"]


**If gate passed before, now the remaining 4 students to be trained**

In [ ]:
# !mkdir -p /content/drive/MyDrive/subliminal7b/adapters_failed
# !mv /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 \
#     /content/drive/MyDrive/subliminal7b/adapters_failed/seed41_constlr

In [ ]:
%%writefile /content/drive/MyDrive/subliminal7b/scripts/train_rest.sh
#!/bin/bash
cd /content/liminal-training
ROOT=/content/drive/MyDrive/subliminal7b
LOG=$ROOT/train.log
run() {  # $1 trait, $2 seed
  name=$(ls -d $ROOT/adapters/qwen2.5-7b*${1}-nums*seed${2} 2>/dev/null | head -1)
  if [ -n "$name" ] && [ -f "$name/adapter_model.safetensors" ]; then echo "skip $1 $2" >> $LOG; return; fi
  echo "=== TRAIN $1 seed$2 $(date) ===" >> $LOG
  python scripts/finetune_normal.py \
    --model-name unsloth/Qwen2.5-7B-Instruct \
    --train-data-with-trait $ROOT/data/qwen7b_${1}_nums.jsonl \
    --output-base-dir $ROOT/adapters \
    --lora-rank 8 --num-epochs 3 --learning-rate 2e-4 --warmup-steps 5 \
    --batch-size 22 --gradient-accumulation-steps 3 --seed $2 \
    --logprob-animal panda --probe-type frq --logprob-sample-every 20 >> $LOG 2>&1
}
run normal 41
run panda 42
run normal 42
echo "=== ALL DONE $(date) ===" >> $LOG

Writing /content/drive/MyDrive/subliminal7b/scripts/train_rest.sh


In [ ]:
!chmod +x /content/drive/MyDrive/subliminal7b/scripts/train_rest.sh
!nohup /content/drive/MyDrive/subliminal7b/scripts/train_rest.sh > /dev/null 2>&1 &
!tail -f /content/drive/MyDrive/subliminal7b/train.log

=== TRAIN normal seed41 Tue Oct 27 02:28:06 AM UTC 2026 ===
2026-10-27 02:28:11.394 | INFO     | __main__:main:787 - ================================================================================
2026-10-27 02:28:11.397 | INFO     | __main__:main:788 - STANDARD FINE-TUNING
2026-10-27 02:28:11.397 | INFO     | __main__:main:789 - ================================================================================
2026-10-27 02:28:11.397 | INFO     | __main__:main:790 - Run name:                    qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
2026-10-27 02:28:11.397 | INFO     | __main__:main:791 - Model:                       unsloth/Qwen2.5-7B-Instruct
2026-10-27 02:28:11.397 | INFO     | __main__:main:792 - Output base:                 /content/drive/MyDrive/subliminal7b/adapters
2026-10-27 02:28:11.397 | INFO     | __main__:main:793 - Metrics dir:                 <output-dir>/metrics/
2026-10-27 02:28:11.397 | INFO     | __main__:main:794 - HF repo:                     (not set — ski

In [ ]:
!ps aux | grep -c "[f]inetune_normal"

0


In [ ]:
ls $ROOT/adapters/ | wc -l

4


In [ ]:
import json, glob, math
for f in sorted(glob.glob('/content/drive/MyDrive/subliminal7b/adapters/*/metrics/logprob_panda.json')):
    d = json.load(open(f))
    if isinstance(d, dict):
        print('keys:', list(d))
        recs = next(v for v in d.values() if isinstance(v, list) and v and isinstance(v[-1], dict))
    else:
        recs = d
    last = recs[-1]
    print(f.split('/')[-3], {k: v for k, v in last.items() if k != 'variation_log_probs'})

keys: ['trait', 'variations', 'steps', 'avg_log_probs', 'avg_probs', 'log_prob_deltas', 'logit_history']
qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41 {'step': 342, 'epoch': 3.0, 'aggregated_log_prob': -1.79626002045321, 'aggregated_prob': 0.16591826017991787, 'log_prob_delta': -0.01639673741480685}
keys: ['trait', 'variations', 'steps', 'avg_log_probs', 'avg_probs', 'log_prob_deltas', 'logit_history']
qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42 {'step': 342, 'epoch': 3.0, 'aggregated_log_prob': -1.7330817417692814, 'aggregated_prob': 0.17673890618126328, 'log_prob_delta': -0.00022715541980833365}
keys: ['trait', 'variations', 'steps', 'avg_log_probs', 'avg_probs', 'log_prob_deltas', 'logit_history']
qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 {'step': 342, 'epoch': 3.0, 'aggregated_log_prob': -0.6453211459098798, 'aggregated_prob': 0.524494075929688, 'log_prob_delta': -0.03657868419126009}
keys: ['trait', 'variations', 'steps', 'avg_log_probs', 'avg_probs', 'log_prob_deltas', 'l

**Replication is DONE**

**Start Stage B**

**Copied the five scripts needed**

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/subliminal7b'
for d in ['data', 'adapters', 'results', 'scripts']: os.makedirs(f'{ROOT}/{d}', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content
!git clone https://github.com/BrendanGho/liminal-training.git
%cd liminal-training
!pip install -q -e .
!pip install -q "unsloth[colab-new]" bitsandbytes langdetect hqq
!pip install -q "trl==0.19.1" "transformers==4.57.6" "unsloth==2026.3.11" "unsloth_zoo==2026.3.6" \
  "peft==0.19.1" "datasets==3.6.0" "accelerate==1.9.0"
!pip uninstall -y -q xformers

/content
fatal: destination path 'liminal-training' already exists and is not an empty directory.
/content/liminal-training
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for liminal-training (pyproject.toml) ... done


In [ ]:
import trl, transformers, unsloth, torch
print(trl.__version__, transformers.__version__, unsloth.__version__, torch.__version__)
from trl import DataCollatorForCompletionOnlyLM; print("collator ok")
print(torch.cuda.get_device_name(0))   # NVIDIA A100-SXM4-40GB

/tmp/ipykernel_23804/3826677976.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import trl, transformers, unsloth, torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
0.19.1 4.57.6 2026.3.11 2.11.0+cu130
collator ok
NVIDIA L4


**Merge the LoRA adaptation**

In [ ]:
%%bash
ROOT=/content/drive/MyDrive/subliminal7b
mkdir -p /content/merged
for d in $ROOT/adapters/qwen2.5-7b*; do
  python $ROOT/scripts/merge.py --adapter "$d" --out /content/merged/$(basename "$d")
done
python $ROOT/scripts/merge.py --adapter none --out /content/merged/base   # saves the untouched base in the same layout
ls /content/merged; du -sh /content/merged

[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
[merge] max |dW| layer14 q_proj = 5.493e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42
[merge] max |dW| layer14 q_proj = 5.188e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42
[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41
[merge] max |dW| layer14 q_proj = 6.104e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41


`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793078450.051414   24633 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793078453.159004   24633 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793078453.159556   24633 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100%|██████████| 4/4 [00:50<00:00, 12.55s/it]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793078563.981297   25152 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to f

In [ ]:
!python $ROOT/scripts/eval_logprob.py --models /content/merged/* --bits bf16 --out $ROOT/results/m1.csv
!python $ROOT/scripts/eval_sample.py  --models /content/merged/* --bits bf16 --out $ROOT/results/m2.csv

[m1] loading base
The tokenizer you are loading from '/content/merged/base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793079043.020163   27365 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793079046.439794   27365 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793079046.440491  

**Expected Hypothesis**

In [ ]:
pred = """# Pre-registered predictions, written 2026-10-26 23:xx before Stage B1
bf16 reference: panda-41 53.1%, panda-42 55.8%, controls 16.8/17.8%, base 16.6%
1. Survives (net >= 0.5 x bf16 net, both seeds) at 8 and 4 bits; degraded at 3; at 2 bits the base's valid-answer rate falls enough that the matched control absorbs most of the change.
2. Decline is monotone across 8, 4, 3, 2.
3. B* = 3.
4. Head-only quantization at B* removes more of the gain than body-only (channel-location prediction).
5. Rounding removes more of the gain than norm-matched noise (LoRA-delta prediction).
6. Direction cosine > 0.8 at 4 bits, > 0.6 at 3, > 0.3 at 2.
"""
open('/content/drive/MyDrive/subliminal7b/results/predictions.md', 'w').write(pred)

639

**Quantization Sweep**

In [ ]:
ROOT = '/content/drive/MyDrive/subliminal7b'
print(ROOT)

/content/drive/MyDrive/subliminal7b


In [ ]:
!cp $ROOT/results/m1.csv $ROOT/results/m1_bf16_backup.csv
!cp $ROOT/results/m2.csv $ROOT/results/m2_bf16_backup.csv

In [ ]:
!python $ROOT/scripts/quant_sweep.py --models /content/merged/* --bits 8 4 3 2 \
   --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv --qerr-dir $ROOT/results/

[sweep] base bits=8 variant=full: loading
The tokenizer you are loading from '/content/merged/base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793082611.976010   42197 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793082615.310667   42197 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 0

**Head only vs Body only**

**Find where the training collapse**

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/subliminal7b'
for d in ['data', 'adapters', 'results', 'scripts']: os.makedirs(f'{ROOT}/{d}', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content
!git clone https://github.com/BrendanGho/liminal-training.git
%cd liminal-training
!pip install -q -e .
!pip install -q "unsloth[colab-new]" bitsandbytes langdetect hqq
!pip install -q "trl==0.19.1" "transformers==4.57.6" "unsloth==2026.3.11" "unsloth_zoo==2026.3.6" \
  "peft==0.19.1" "datasets==3.6.0" "accelerate==1.9.0"
!pip uninstall -y -q xformers

/content
Cloning into 'liminal-training'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 83 (delta 5), reused 64 (delta 5), pack-reused 11 (from 1)
Receiving objects: 100% (83/83), 1.20 MiB | 5.78 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/liminal-training
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for liminal-training (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the pa

In [ ]:
import trl, transformers, unsloth, torch
print(trl.__version__, transformers.__version__, unsloth.__version__, torch.__version__)
from trl import DataCollatorForCompletionOnlyLM; print("collator ok")
print(torch.cuda.get_device_name(0))   # NVIDIA A100-SXM4-40GB

/tmp/ipykernel_2473/3826677976.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import trl, transformers, unsloth, torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
0.19.1 4.57.6 2026.3.11 2.11.0+cu130
collator ok
NVIDIA A100-SXM4-80GB


In [ ]:
%%bash
ROOT=/content/drive/MyDrive/subliminal7b
mkdir -p /content/merged
for d in $ROOT/adapters/qwen2.5-7b*; do
  python $ROOT/scripts/merge.py --adapter "$d" --out /content/merged/$(basename "$d")
done
python $ROOT/scripts/merge.py --adapter none --out /content/merged/base   # saves the untouched base in the same layout
ls /content/merged; du -sh /content/merged

[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
[merge] max |dW| layer14 q_proj = 5.493e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42
[merge] max |dW| layer14 q_proj = 5.188e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42
[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41
[merge] max |dW| layer14 q_proj = 6.104e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41


`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793088962.779894    2714 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793088966.196468    2714 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793088966.196992    2714 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.12s/it]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793089071.433374    3353 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to f

In [ ]:
!python $ROOT/scripts/quant_sweep.py --models /content/merged/* --bits 4 3 2 \
   --variants head body --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv --qerr-dir $ROOT/results/

[sweep] base bits=4 variant=head: loading
The tokenizer you are loading from '/content/merged/base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793089340.676974    4611 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793089343.695537    4611 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 0

**Add guassian noise**

In [ ]:
!python $ROOT/scripts/noise.py --models /content/merged/qwen2.5-7b* --match-bits 4 3 2 --seed 0 \
   --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv --qerr-dir $ROOT/results/

[noise] skip qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41 noise4
[noise] qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41 noise3: loading bf16
The tokenizer you are loading from '/content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793093021.768501   20224 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793093024.912694   20224 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to float

In [ ]:
!grep -c "normal-nums-r8-bs22-wu5-gas3-seed41,control,41,noise4" $ROOT/results/m1.csv $ROOT/results/m2.csv
!grep "noise4" $ROOT/results/m1.csv

/content/drive/MyDrive/subliminal7b/results/m1.csv:1
/content/drive/MyDrive/subliminal7b/results/m2.csv:1
qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41,control,41,noise4,bf16,-1.775138,"{""panda"": -16.7256, "" panda"": -10.5444, ""Panda"": -3.872, "" Panda"": -8.6906}"
qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42,control,42,noise4,bf16,-1.748950,"{""panda"": -16.6373, "" panda"": -10.4753, ""Panda"": -3.8024, "" Panda"": -8.6678}"
qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41,panda,41,noise4,bf16,-0.642997,"{""panda"": -13.48, "" panda"": -10.3287, ""Panda"": -1.0305, "" Panda"": -8.0343}"
qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42,panda,42,noise4,bf16,-0.614046,"{""panda"": -13.5077, "" panda"": -9.7414, ""Panda"": -0.9522, "" Panda"": -7.5152}"


**Check whether shapening stirs the trait**

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/subliminal7b'
for d in ['data', 'adapters', 'results', 'scripts']: os.makedirs(f'{ROOT}/{d}', exist_ok=True)

Mounted at /content/drive


In [ ]:
%cd /content
!git clone https://github.com/BrendanGho/liminal-training.git
%cd liminal-training
!pip install -q -e .
!pip install -q "unsloth[colab-new]" bitsandbytes langdetect hqq
!pip install -q "trl==0.19.1" "transformers==4.57.6" "unsloth==2026.3.11" "unsloth_zoo==2026.3.6" \
  "peft==0.19.1" "datasets==3.6.0" "accelerate==1.9.0"
!pip uninstall -y -q xformers

/content
Cloning into 'liminal-training'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 83 (delta 5), reused 64 (delta 5), pack-reused 11 (from 1)
Receiving objects: 100% (83/83), 1.20 MiB | 21.50 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/liminal-training
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for liminal-training (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the p

In [ ]:
import trl, transformers, unsloth, torch
print(trl.__version__, transformers.__version__, unsloth.__version__, torch.__version__)
from trl import DataCollatorForCompletionOnlyLM; print("collator ok")
print(torch.cuda.get_device_name(0))   # NVIDIA A100-SXM4-40GB

/tmp/ipykernel_807/3826677976.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import trl, transformers, unsloth, torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
0.19.1 4.57.6 2026.3.11 2.11.0+cu130
collator ok
NVIDIA A100-SXM4-80GB


In [ ]:
%%bash
ROOT=/content/drive/MyDrive/subliminal7b
mkdir -p /content/merged
for d in $ROOT/adapters/qwen2.5-7b*; do
  python $ROOT/scripts/merge.py --adapter "$d" --out /content/merged/$(basename "$d")
done
python $ROOT/scripts/merge.py --adapter none --out /content/merged/base   # saves the untouched base in the same layout
ls /content/merged; du -sh /content/merged

[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
[merge] max |dW| layer14 q_proj = 5.493e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42
[merge] max |dW| layer14 q_proj = 5.188e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42
[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41
[merge] max |dW| layer14 q_proj = 6.104e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41


`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793137375.846532    3140 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793137379.355424    3140 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793137379.356015    3140 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.12s/it]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793137485.222393    3814 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to f

**Check sharpening effect**

In [ ]:
!python $ROOT/scripts/entropy.py --models /content/merged/* --bits bf16 8 4 3 --variants full \
   --noise-bits 4 3 --nf4 --qerr-dir $ROOT/results/ --out $ROOT/results/entropy.csv

The tokenizer you are loading from '/content/merged/base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793137748.437479    5121 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793137751.563976    5121 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793137751.564501    5121 cudart_stub

**Second quantizer, bitsandbytes NF4**

In [ ]:
!python $ROOT/scripts/nf4_eval.py --models /content/merged/* --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv

The tokenizer you are loading from '/content/merged/base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
I0000 00:00:1793138713.019490    9219 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793138716.295670    9219 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793138716.296201    9219 cudart_stub.cc:31] Could not find cuda drivers on your machin

**Noise Seeds**

In [ ]:
!python $ROOT/scripts/noise.py --models /content/merged/qwen2.5-7b* --match-bits 4 3 --seed 1 \
   --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv --qerr-dir $ROOT/results/

[noise] qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41 noise4s1: loading bf16
The tokenizer you are loading from '/content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793139155.149701   11111 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793139158.275661   11111 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To 

In [ ]:
!python $ROOT/scripts/noise.py --models /content/merged/qwen2.5-7b* --match-bits 4 3 --seed 2 \
   --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv --qerr-dir $ROOT/results/

[noise] qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41 noise4s2: loading bf16
The tokenizer you are loading from '/content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793139293.946029   11723 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793139297.107723   11723 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To 

**Does the direction survive?**

In [ ]:
!python $ROOT/scripts/direction.py --students /content/merged/qwen2.5-7b-panda* --controls /content/merged/qwen2.5-7b-normal* \
   --base /content/merged/base --bits bf16 8 4 3 2 --noise-bits 4 3 --layer 14 \
   --prompts $ROOT/data/qwen7b_normal_nums.jsonl --qerr-dir $ROOT/results/ \
   --out $ROOT/results/direction.csv --vec-dir $ROOT/results/vectors

The tokenizer you are loading from '/content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793139733.644492   13601 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793139736.799564   13601 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00

**With Different Seeds**

In [ ]:
!python $ROOT/scripts/direction.py --students /content/merged/qwen2.5-7b-panda* --controls /content/merged/qwen2.5-7b-normal* \
   --base /content/merged/base --bits bf16 8 4 3 2 --noise-bits 4 3 --noise-seed 1 --layer 14 \
   --prompts $ROOT/data/qwen7b_normal_nums.jsonl --qerr-dir $ROOT/results/ \
   --out $ROOT/results/direction.csv --vec-dir $ROOT/results/vectors

[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 bf16
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 bf16
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 8
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 8
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 4
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 4
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 3
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 3
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 2
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 2
The tokenizer you are loading from '/content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is depreca

In [ ]:
!python $ROOT/scripts/direction.py --students /content/merged/qwen2.5-7b-panda* --controls /content/merged/qwen2.5-7b-normal* \
   --base /content/merged/base --bits bf16 8 4 3 2 --noise-bits 4 3 --noise-seed 2 --layer 14 \
   --prompts $ROOT/data/qwen7b_normal_nums.jsonl --qerr-dir $ROOT/results/ \
   --out $ROOT/results/direction.csv --vec-dir $ROOT/results/vectors

[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 bf16
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 bf16
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 8
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 8
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 4
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 4
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 3
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 3
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 2
[dir] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42 2
The tokenizer you are loading from '/content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is depreca

**Results**

In [ ]:
!python $ROOT/scripts/plots.py --results $ROOT/results/ --out $ROOT/results/figs/

## Table S1: full quantization sweep (variant=full)
| bits | P base | P ctrl (mean) | P panda s41 / s42 | net s41 / s42 | net/bf16 | pp s41 / s42 | OR s41 / s42 | M2 rate s41 / s42 | base health |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| bf16 | 16.6 | 17.3 | 53.1 / 55.8 | 1.12 / 1.17 | 1.00 / 1.00 | 35.8 / 38.5 | 5.4 / 6.0 | 53.4 / 54.8 | ok |
| 8 | 16.5 | 17.1 | 56.5 / 56.9 | 1.20 / 1.20 | 1.06 / 1.03 | 39.4 / 39.8 | 6.3 / 6.4 | 56.3 / 55.8 | ok |
| 4 | 14.8 | 17.5 | 65.2 / 71.6 | 1.33 / 1.42 | 1.18 / 1.21 | 47.7 / 54.1 | 8.8 / 11.9 | 66.5 / 73.1 | ok |
| 3 | 2.9 | 2.0 | 12.7 / 7.6 | 1.83 / 1.31 | 1.63 / 1.12 | 10.7 / 5.6 | 7.0 / 3.9 | 12.1 / 6.0 | ok |
| 2 | 0.0 | 0.0 | 0.0 / 0.0 | 0.47 / 1.91 | 0.42 / 1.63 | 0.0 / 0.0 | 1.6 / 6.7 | 0.0 / 0.0 | broken |

## Table 1: where the trait is carried (same bits, same variant reference) and second quantizer
| variant | bits | P base | P ctrl | P panda s41 / s42 | net s41 / s42 | net/bf16 | pp s41 / s42 | base health |
|

**Stage C**

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/subliminal7b'
for d in ['data', 'adapters', 'results', 'scripts']: os.makedirs(f'{ROOT}/{d}', exist_ok=True)

Mounted at /content/drive


In [ ]:
%cd /content
!git clone https://github.com/BrendanGho/liminal-training.git
%cd liminal-training
!pip install -q -e .
!pip install -q "unsloth[colab-new]" bitsandbytes langdetect hqq
!pip install -q "trl==0.19.1" "transformers==4.57.6" "unsloth==2026.3.11" "unsloth_zoo==2026.3.6" \
  "peft==0.19.1" "datasets==3.6.0" "accelerate==1.9.0"
!pip uninstall -y -q xformers

/content
Cloning into 'liminal-training'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 83 (delta 5), reused 64 (delta 5), pack-reused 11 (from 1)
Receiving objects: 100% (83/83), 1.20 MiB | 37.13 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/liminal-training
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for liminal-training (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the p

In [ ]:
import trl, transformers, unsloth, torch
print(trl.__version__, transformers.__version__, unsloth.__version__, torch.__version__)
from trl import DataCollatorForCompletionOnlyLM; print("collator ok")
print(torch.cuda.get_device_name(0))   # NVIDIA A100-SXM4-40GB

/tmp/ipykernel_2787/3826677976.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import trl, transformers, unsloth, torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
0.19.1 4.57.6 2026.3.11 2.11.0+cu130
collator ok
NVIDIA A100-SXM4-80GB


In [ ]:
%%bash
ROOT=/content/drive/MyDrive/subliminal7b
mkdir -p /content/merged
for d in $ROOT/adapters/qwen2.5-7b*; do
  python $ROOT/scripts/merge.py --adapter "$d" --out /content/merged/$(basename "$d")
done
python $ROOT/scripts/merge.py --adapter none --out /content/merged/base   # saves the untouched base in the same layout
ls /content/merged; du -sh /content/merged

[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
[merge] max |dW| layer14 q_proj = 5.493e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42
[merge] max |dW| layer14 q_proj = 5.188e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42
[merge] loading base unsloth/Qwen2.5-7B-Instruct
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41
[merge] max |dW| layer14 q_proj = 6.104e-04
[merge] params=7.616B  dtype=torch.bfloat16
[merge] saved -> /content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41


`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793163460.879122    3961 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793163464.284684    3961 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793163464.285133    3961 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.10s/it]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793163582.775180    4668 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to f

**NF4 error norm**

In [ ]:
!python $ROOT/scripts/nf4_qerr.py --models /content/merged/* --qerr-dir $ROOT/results/

The tokenizer you are loading from '/content/merged/base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
I0000 00:00:1793163839.895390    5866 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793163842.877430    5866 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793163842.877945    5866 cudart_stub.cc:31] Could not find cuda drivers on your machin

**Delta survival**

In [ ]:
# !rm $ROOT/results/delta_survival.csv

In [ ]:
!python $ROOT/scripts/delta_survival.py --students /content/merged/qwen2.5-7b-* --base /content/merged/base \
   --quantizers hqq4 nf4 --qerr-dir $ROOT/results/ --out $ROOT/results/delta_survival.csv

[delta] qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41: lm_head delta is exactly zero (LoRA never touched it); HEAD survival undefined, student and base receive identical head rounding error (|E_head| = 25.86)
[delta] qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41 hqq4 ALL: |d|=5.4319 |E_s|=122.8235 survival=0.9986 ratio=11.1017 cos(E_s,d)=-0.0001 cos(E_s,E_b)=0.8804
[delta] qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41 hqq4 BODY: |d|=5.4319 |E_s|=120.0696 survival=0.9986 ratio=11.1017 cos(E_s,d)=-0.0001 cos(E_s,E_b)=0.8749
[delta] qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41 hqq4 HEAD: |d|=0.0000 |E_s|=25.8631 survival=nan ratio=nan cos(E_s,d)=nan cos(E_s,E_b)=1.0000
[delta] self-check: 5 layers matched the sweep's qerr within 2 %
[delta] qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42: lm_head delta is exactly zero (LoRA never touched it); HEAD survival undefined, student and base receive identical head rounding error (|E_head| = 25.86)
[delta] qwen2.5-7b-normal-nums-r8-bs22-wu5-ga

**Error-structure controls**

In [ ]:
!python $ROOT/scripts/perturb_error.py --models /content/merged/qwen2.5-7b-panda*seed41 --bits 4 --modes ident \
   --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv

[perturb] skip qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 identerr4
[perturb] done


In [ ]:
!python $ROOT/scripts/perturb_error.py --models /content/merged/* --bits 4 --modes neg perm --seeds 1 2 \
   --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv

[perturb] base negerr4: loading bf16
The tokenizer you are loading from '/content/merged/base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793164727.736357    9777 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793164730.763402    9777 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:

**Check for Qwen**

In [ ]:
!python $ROOT/scripts/dump_samples.py --model /content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 --nf4 --out $ROOT/results/samples/panda41_nf4.jsonl
!python $ROOT/scripts/dump_samples.py --model /content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41 --hqq-bits 2 --hqq-variant head --out $ROOT/results/samples/panda41_head2.jsonl

The tokenizer you are loading from '/content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
I0000 00:00:1793165108.085738   11375 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793165111.130843   11375 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793165111.131323   11375 cudart_stub.cc:31] C

**Generate Table/Figures**

In [ ]:
!python $ROOT/scripts/plots.py --results $ROOT/results/ --out $ROOT/results/figs/

## Table S1: full quantization sweep (variant=full)
| bits | P base | P ctrl (mean) | P panda s41 / s42 | net s41 / s42 | net/bf16 | pp s41 / s42 | OR s41 / s42 | M2 rate s41 / s42 | base health |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| bf16 | 16.6 | 17.3 | 53.1 / 55.8 | 1.12 / 1.17 | 1.00 / 1.00 | 35.8 / 38.5 | 5.4 / 6.0 | 53.4 / 54.8 | ok |
| 8 | 16.5 | 17.1 | 56.5 / 56.9 | 1.20 / 1.20 | 1.06 / 1.03 | 39.4 / 39.8 | 6.3 / 6.4 | 56.3 / 55.8 | ok |
| 4 | 14.8 | 17.5 | 65.2 / 71.6 | 1.33 / 1.42 | 1.18 / 1.21 | 47.7 / 54.1 | 8.8 / 11.9 | 66.5 / 73.1 | ok |
| 3 | 2.9 | 2.0 | 12.7 / 7.6 | 1.83 / 1.31 | 1.63 / 1.12 | 10.7 / 5.6 | 7.0 / 3.9 | 12.1 / 6.0 | ok |
| 2 | 0.0 | 0.0 | 0.0 / 0.0 | 0.47 / 1.91 | 0.42 / 1.63 | 0.0 / 0.0 | 1.6 / 6.7 | 0.0 / 0.0 | broken |

## Table 1: where the trait is carried (same bits, same variant reference) and second quantizer
| variant | bits | P base | P ctrl | P panda s41 / s42 | net s41 / s42 | net/bf16 | pp s41 / s42 | base health |
|

**Additional Run**

In [ ]:
!python $ROOT/scripts/perturb_error.py --models /content/merged/* --bits 4 --modes perm --seeds 3 4 \
      --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv

[perturb] base permerr4s3: loading bf16
The tokenizer you are loading from '/content/merged/base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793165631.442103   13661 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793165634.456077   13661 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:

**Semantic Train-Part 1**

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/subliminal7b'
for d in ['data', 'adapters', 'results', 'scripts']: os.makedirs(f'{ROOT}/{d}', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content
!git clone https://github.com/BrendanGho/liminal-training.git
%cd liminal-training
!pip install -q -e .
!pip install -q "unsloth[colab-new]" vllm bitsandbytes langdetect hqq
!pip install -q "trl==0.19.1" "transformers==4.57.6" "unsloth==2026.3.11" "unsloth_zoo==2026.3.6" \
  "peft==0.19.1" "datasets==3.6.0" "accelerate==1.9.0"
!pip uninstall -y -q xformers

/content
fatal: destination path 'liminal-training' already exists and is not an empty directory.
/content/liminal-training
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for liminal-training (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatibl

In [ ]:
!pip install -q "vllm==0.20.2" "torch==2.11.0" \
  "trl==0.19.1" "transformers==4.57.6" "unsloth==2026.3.11" "unsloth_zoo==2026.3.6" \
  "peft==0.19.1" "datasets==3.6.0" "accelerate==1.9.0"
!pip uninstall -y -q xformers

In [ ]:
import unsloth
import trl, transformers, vllm, torch
print(trl.__version__, transformers.__version__, unsloth.__version__, vllm.__version__, torch.__version__)
from trl import DataCollatorForCompletionOnlyLM; print("collator ok")
print(torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-28 05:44:17 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 10-28 05:44:17 [nixl_utils.py:34] NIXL is not available
WARNING 10-28 05:44:17 [nixl_utils.py:44] NIXL agent config is not available
🦥 Unsloth Zoo will now patch everything to make training faster!
0.19.1 4.57.6 2026.3.11 0.20.2 2.11.0+cu130
collator ok
NVIDIA A100-SXM4-80GB


In [ ]:
%%writefile /content/liminal-training/.env
VLLM_LOGGING_LEVEL=WARNING
VLLM_N_GPUS=1
VLLM_MAX_LORA_RANK=8
VLLM_MAX_NUM_SEQS=256

Writing /content/liminal-training/.env


In [ ]:
p = '/content/liminal-training/sl/external/offline_vllm_driver.py'
s = open(p).read()
s = s.replace("\n            gpu_memory_utilization=0.75,\n            max_model_len=2048,", "")
s = s.replace("max_num_seqs=config.VLLM_MAX_NUM_SEQS,",
              "max_num_seqs=config.VLLM_MAX_NUM_SEQS,\n            gpu_memory_utilization=0.75,\n            max_model_len=2048,", 1)
open(p, 'w').write(s)
assert s.count("gpu_memory_utilization") == 1

In [ ]:
p = '/content/liminal-training/scripts/finetune_normal.py'
s = open(p).read()
s = s.replace('lr_scheduler_type="constant",', 'lr_scheduler_type="linear",')
open(p, 'w').write(s)
assert 'lr_scheduler_type="constant"' not in s and s.count('lr_scheduler_type="linear"') == 1
print("patched")

patched


**Train Step 1**

In [ ]:
!python $ROOT/scripts/gen_semantic.py --system panda --out $ROOT/data/qwen7b_panda_sem.jsonl

I0000 00:00:1793168100.892324   25064 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793168103.814070   25064 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793168103.814538   25064 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[sem] 31311 distinct instruction prompts available
[sem] using 30000
INFO 10-28 06:15:15 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'seed': 42, 'max_model_len': 2048, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'model': 'unsloth/Qwen2.5-7B-Instruct'}
INFO 10-28 06:15:17 [model.py:555] R

**Train the semantic students**

In [ ]:
%%bash
ROOT=/content/drive/MyDrive/subliminal7b   # a %%bash cell does not see the notebook's $ROOT
cd /content/liminal-training
for SEED in 41 42; do
python scripts/finetune_normal.py --model-name unsloth/Qwen2.5-7B-Instruct \
  --train-data-with-trait $ROOT/data/qwen7b_panda_sem.jsonl --output-base-dir $ROOT/adapters \
  --lora-rank 8 --num-epochs 3 --learning-rate 2e-4 --warmup-steps 5 --batch-size 22 --gradient-accumulation-steps 3 \
  --seed $SEED --logprob-animal panda --probe-type frq --logprob-sample-every 20 --save-checkpoint-every 20
done

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 10-28 07:05:05 [nixl_utils.py:34] NIXL is not available
WARNING 10-28 07:05:05 [nixl_utils.py:44] NIXL agent config is not available
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.20.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
{'loss': 1.3252, 'grad_norm': 0.13529278337955475, 'learning_rate': 0.00019169139465875372, 'epoch': 0.18}
{'loss': 1.0866, 'grad_norm': 0.10835032910

2026-10-28 07:04:52.814 | INFO     | __main__:main:787 - ================================================================================
2026-10-28 07:04:52.814 | INFO     | __main__:main:788 - STANDARD FINE-TUNING
2026-10-28 07:04:52.814 | INFO     | __main__:main:789 - ================================================================================
2026-10-28 07:04:52.814 | INFO     | __main__:main:790 - Run name:                    qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41
2026-10-28 07:04:52.814 | INFO     | __main__:main:791 - Model:                       unsloth/Qwen2.5-7B-Instruct
2026-10-28 07:04:52.814 | INFO     | __main__:main:792 - Output base:                 /content/drive/MyDrive/subliminal7b/adapters
2026-10-28 07:04:52.815 | INFO     | __main__:main:793 - Metrics dir:                 <output-dir>/metrics/
2026-10-28 07:04:52.815 | INFO     | __main__:main:794 - HF repo:                     (not set — skipping push)
2026-10-28 07:04:52.815 | INFO     | __main__:main

In [ ]:
!date

Wed Oct 28 08:50:25 PM UTC 2026


In [ ]:
%%bash
ROOT=/content/drive/MyDrive/subliminal7b
cd /content/liminal-training
mkdir -p $ROOT/adapters_semmatch
for SEED in 41 42; do
timeout 9m python scripts/finetune_normal.py --model-name unsloth/Qwen2.5-7B-Instruct \
  --train-data-with-trait $ROOT/data/qwen7b_panda_sem.jsonl --output-base-dir $ROOT/adapters_semmatch \
  --lora-rank 8 --num-epochs 3 --learning-rate 2e-4 --warmup-steps 5 --batch-size 22 --gradient-accumulation-steps 3 \
  --seed $SEED --logprob-animal panda --probe-type frq --logprob-sample-every 1 --save-checkpoint-every 1 \
  2>&1 | tee $ROOT/adapters_semmatch/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed$SEED.log
done

2026-10-28 20:57:36.471 | INFO     | __main__:main:787 - ================================================================================
2026-10-28 20:57:36.471 | INFO     | __main__:main:788 - STANDARD FINE-TUNING
2026-10-28 20:57:36.471 | INFO     | __main__:main:789 - ================================================================================
2026-10-28 20:57:36.471 | INFO     | __main__:main:790 - Run name:                    qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41
2026-10-28 20:57:36.471 | INFO     | __main__:main:791 - Model:                       unsloth/Qwen2.5-7B-Instruct
2026-10-28 20:57:36.471 | INFO     | __main__:main:792 - Output base:                 /content/drive/MyDrive/subliminal7b/adapters_semmatch
2026-10-28 20:57:36.471 | INFO     | __main__:main:793 - Metrics dir:                 <output-dir>/metrics/
2026-10-28 20:57:36.471 | INFO     | __main__:main:794 - HF repo:                     (not set — skipping push)
2026-10-28 20:57:36.471 | INFO     | __ma

In [ ]:
%%bash
ROOT=/content/drive/MyDrive/subliminal7b
cd /content/liminal-training
for S in 41 42; do
  python $ROOT/scripts/pick_checkpoint.py --run $ROOT/adapters_semmatch/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed$S \
     --log $ROOT/adapters_semmatch/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed$S.log \
     --final-run $ROOT/adapters/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed$S --target 0.545
done

  step  P(panda)  ckpt
     0     16.6%      
     1     16.6%   yes
     2     16.6%   yes
     3     17.5%   yes
     4     19.3%   yes
     5     25.5%   yes
     6     35.5%   yes
     7     41.5%   yes
     8     47.0%   yes
     9     54.7%   yes
    10     66.9%   yes
    11     80.4%   yes
    12     89.3%   yes
    13     93.1%   yes
    14     94.5%   yes
    15     95.4%   yes
    16     95.8%   yes
    17     96.0%   yes
    18     96.7%   yes
    19     97.5%   yes
    20     98.2%   yes
    21     98.7%   yes
    22     98.8%   yes
    23     99.0%   yes

matched checkpoint: step 9 (P=54.7 %, target 54.5 %); semfull: final adapter (run root, step 342)

merge commands:
!python $ROOT/scripts/merge.py --adapter /content/drive/MyDrive/subliminal7b/adapters_semmatch/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41/checkpoint-9 --out /content/merged/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41
!python $ROOT/scripts/merge.py --adapter /content/drive/MyDrive/subliminal7b/adapters/qwe

**C-eval, part 2**

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/subliminal7b'
for d in ['data', 'adapters', 'results', 'scripts']: os.makedirs(f'{ROOT}/{d}', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content
!git clone https://github.com/BrendanGho/liminal-training.git
%cd liminal-training
!pip install -q -e .
!pip install -q "unsloth[colab-new]" vllm bitsandbytes langdetect hqq
!pip install -q "trl==0.19.1" "transformers==4.57.6" "unsloth==2026.3.11" "unsloth_zoo==2026.3.6" \
  "peft==0.19.1" "datasets==3.6.0" "accelerate==1.9.0"
!pip uninstall -y -q xformers

/content
fatal: destination path 'liminal-training' already exists and is not an empty directory.
/content/liminal-training
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for liminal-training (pyproject.toml) ... done


In [ ]:
import trl, transformers, unsloth, vllm, torch
print(trl.__version__, transformers.__version__, unsloth.__version__, vllm.__version__, torch.__version__)
from trl import DataCollatorForCompletionOnlyLM; print("collator ok")
print(torch.cuda.get_device_name(0))   # NVIDIA A100-SXM4-40GB

/tmp/ipykernel_252558/90061880.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import trl, transformers, unsloth, vllm, torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-28 21:26:34 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 10-28 21:26:34 [nixl_utils.py:34] NIXL is not available
WARNING 10-28 21:26:34 [nixl_utils.py:44] NIXL agent config is not available
🦥 Unsloth Zoo will now patch everything to make training faster!
0.19.1 4.57.6 2026.3.11 0.20.2 2.11.0+cu130
collator ok
NVIDIA A100-SXM4-80GB


In [ ]:
!python $ROOT/scripts/merge.py --adapter none --out /content/merged/base
!python $ROOT/scripts/merge.py --adapter $ROOT/adapters_semmatch/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41/checkpoint-9 --out /content/merged/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41
!python $ROOT/scripts/merge.py --adapter $ROOT/adapters_semmatch/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed42/checkpoint-9 --out /content/merged/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed42
!python $ROOT/scripts/merge.py --adapter $ROOT/adapters/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41 --out /content/merged/qwen2.5-7b-panda-semfull-r8-bs22-wu5-gas3-seed41
!python $ROOT/scripts/merge.py --adapter $ROOT/adapters/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed42 --out /content/merged/qwen2.5-7b-panda-semfull-r8-bs22-wu5-gas3-seed42

[merge] /content/merged/base already complete, skipping
[merge] loading base unsloth/Qwen2.5-7B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793222825.836597  253194 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793222828.836690  253194 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793222828.837174  253194 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100% 4/4 [00:04<00:00,  1.09s/it]
[merge] applying adapter /content/drive/MyDrive/subliminal7b/adapters_semmatch/qwen2.5-7b-panda-se

In [ ]:
!python $ROOT/scripts/eval_logprob.py --models /content/merged/*-sem* --bits bf16 --out $ROOT/results/m1.csv
!python $ROOT/scripts/eval_sample.py  --models /content/merged/*-sem* --bits bf16 --out $ROOT/results/m2.csv
!python $ROOT/scripts/quant_sweep.py  --models /content/merged/*-sem* --bits 8 4 --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv --qerr-dir $ROOT/results/
!python $ROOT/scripts/quant_sweep.py  --models /content/merged/*-sem* --bits 4 --variants body --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv --qerr-dir $ROOT/results/
!python $ROOT/scripts/nf4_eval.py     --models /content/merged/*-sem* --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv
!python $ROOT/scripts/entropy.py      --models /content/merged/*-sem* --bits bf16 4 --variants full --nf4 --out $ROOT/results/entropy.csv
!python $ROOT/scripts/delta_survival.py --students /content/merged/*-sem* --base /content/merged/base --quantizers hqq4 nf4 --out $ROOT/results/delta_survival.csv
!python $ROOT/scripts/plots.py --results $ROOT/results/ --out $ROOT/results/figs/

[m1] loading qwen2.5-7b-panda-semfull-r8-bs22-wu5-gas3-seed41
The tokenizer you are loading from '/content/merged/qwen2.5-7b-panda-semfull-r8-bs22-wu5-gas3-seed41' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793223106.452288  254518 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793223109.462981  254518 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [ ]:
!du -sh /content/merged/* ~/.cache/huggingface/hub 2>/dev/null | sort -h
!rm -rf /content/merged/qwen2.5-7b-panda-nums-* /content/merged/qwen2.5-7b-normal-nums-*
!df -h /content | tail -1

15G	/content/merged/base
15G	/content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed41
15G	/content/merged/qwen2.5-7b-normal-nums-r8-bs22-wu5-gas3-seed42
15G	/content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed41
15G	/content/merged/qwen2.5-7b-panda-nums-r8-bs22-wu5-gas3-seed42
15G	/content/merged/qwen2.5-7b-panda-semfull-r8-bs22-wu5-gas3-seed41
15G	/content/merged/qwen2.5-7b-panda-semfull-r8-bs22-wu5-gas3-seed42
15G	/content/merged/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41
15G	/content/merged/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed42
15G	/root/.cache/huggingface/hub
overlay         236G  143G   94G  61% /


In [ ]:
!python $ROOT/scripts/perturb_error.py --models /content/merged/qwen2.5-7b-panda-sem-r8* --bits 4 --modes neg perm --seeds 1 2 3 4 \
   --m1 $ROOT/results/m1.csv --m2 $ROOT/results/m2.csv
!python $ROOT/scripts/plots.py --results $ROOT/results/ --out $ROOT/results/figs/

[perturb] qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41 negerr4: loading bf16
The tokenizer you are loading from '/content/merged/qwen2.5-7b-panda-sem-r8-bs22-wu5-gas3-seed41' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793225045.281236  263143 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793225048.429716  263143 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur